# Distillation Prompts Usage Example

This notebook demonstrates how to use the LangChain runnables defined in `src/analyzer/prompts/distillation/` to analyze code against CFE/RACFE rules defined in `dev_docs/rules.md`.

## 1. Setup and Imports

First, we need to import necessary libraries and add the project root to the Python path so we can import our custom modules.

In [7]:
import sys
import os
import json
import ast  # For safely evaluating the byte string literal
from pathlib import Path
from typing import Any, List, Optional, Dict, Type

# Add project root to sys.path to allow importing src modules
try:
    # Adjust based on notebook location relative to workspace root
    # Since this notebook is in src/analyzer, the root is 2 levels up
    project_root = Path.cwd().parent.parent
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    print(f"Project root added to sys.path: {project_root}")
except Exception as e:
    print(f"Error adding project root: {e}")

# Langchain imports
from langchain_core.language_models.llms import LLM
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.runnables import Runnable 
from langchain_core.prompts import PromptTemplate # <<< Added import
from langchain_core.output_parsers import StrOutputParser # <<< Added import
from dotenv import load_dotenv 
from pydantic import BaseModel 

# Import our custom modules (prompts, models, utils)
try:
    # Note: Ensure the kernel's environment has access to these paths
    from src.analyzer.prompts.distillation.models import (
        GraphAnalysisOutput,
        ProtectionDeterminationOutput,
        RuleVerificationOutput,
        ComplianceReport
    )
    # Import the runnable creation functions (which return DistillationPrompt objects)
    from src.analyzer.prompts.distillation.graph_analysis import create_graph_analysis_runnable
    from src.analyzer.prompts.distillation.protection_determination import create_protection_determination_runnable
    from src.analyzer.prompts.distillation.rule_verification import create_rule_verification_runnable
    from src.analyzer.prompts.distillation.reporting import create_reporting_runnable
    # Import the container type itself
    from src.analyzer.prompts.distillation.prompt_types import DistillationPrompt 
    # Import utils
    from src.analyzer.data_util.file_utils import load_rules
    from src.analyzer.data_util.key_info_extraction import extract_core_instructions 
    # Import the real LLM getter function
    from src.analyzer.service.llm_service import get_langchain_llm 
except ImportError as e:
    print(f"Import Error: {e}")
    print("Make sure the project structure is correct and the notebook kernel has access.")

Project root added to sys.path: /workspace


## 2. Initialize Real LLM

This step initializes the `ChatDeepSeek` model using the function from `llm_service.py`. 
**Important:** Ensure your `DEEPSEEK_API_KEY` environment variable is set, typically in a `.env` file in the project root (`/workspace/data/.env`).

In [8]:
# Load environment variables from .env file if it exists
dotenv_path = project_root / 'data' / '.env' # Corrected path
if dotenv_path.exists():
    print(f"Loading environment variables from: {dotenv_path}")
    # Use override=True if you want .env to take precedence over system env vars
    load_dotenv(dotenv_path=dotenv_path, override=True)
else:
    print(f"Warning: .env file not found at {dotenv_path}. Ensure DEEPSEEK_API_KEY is set globally or manually.")

# Initialize the LLM using the service function
llm = None # Initialize llm to None
try:
    llm = get_langchain_llm()
    print(f"LLM Initialized: {type(llm)}")
except ValueError as e: # Catch API key error specifically
    print(f"LLM Initialization Error: {e}")
    print("Please ensure DEEPSEEK_API_KEY is set in your environment or .env file.")
except ImportError as e:
    print(f"Import Error for LLM Provider: {e}. Is langchain-deepseek installed?")
except Exception as e: # Catch any other unexpected errors
    print(f"An unexpected error occurred during LLM initialization: {e}")

Loading environment variables from: /workspace/data/.env
LLM Initialized: <class 'langchain_deepseek.chat_models.ChatDeepSeek'>


## 3. Load and Prepare Example Data from record.json

Load `src/analyzer/record.json`, extract the RTL content, and process it using `extract_core_instructions`.

In [10]:
# Path to the record file
record_file_path = project_root / "src" / "analyzer" / "record.json"

# Key for the specific RTL file to extract (e.g., from 'makeStrings')
# Change this key if you want to analyze a different RTL file from the record
rtl_key = "makeStrings/RTL_Protected.txt" # Or "makeStrings/RTL_Protected.txt"

example_code_raw = None
example_code = None # This will hold the processed code
try:
    print(f"Loading record data from: {record_file_path}")
    with open(record_file_path, 'r', encoding='utf-8') as f:
        record_data = json.load(f)
    
    # Extract the byte string literal content
    content_str = record_data["plugin_files"][rtl_key]["content"]
    
    # Safely evaluate the literal to get the actual byte string
    rtl_bytes = ast.literal_eval(content_str)
    
    # Decode the byte string to a normal string (assuming utf-8)
    example_code_raw = rtl_bytes.decode('utf-8', errors='ignore') # Use ignore for potential decoding errors
    print(f"Successfully extracted and decoded RAW RTL content for key: {rtl_key}")
    print("--- Raw RTL Snippet ---")
    print(example_code_raw[:500] + ('...' if len(example_code_raw) > 500 else ''))
    print("-----------------------")
    
    # Extract core instructions
    print("Extracting core instructions from loaded RTL...")
    example_code = extract_core_instructions(example_code_raw)
    print(f"Original raw size: {len(example_code_raw)} chars, Processed size: {len(example_code)} chars")
    print("--- Processed RTL Snippet ---")
    print(example_code[:500] + ('...' if len(example_code) > 500 else ''))
    print("---------------------------")
    
except FileNotFoundError:
    print(f"Error: Record file not found at {record_file_path}")
except KeyError:
    print(f"Error: Key '{rtl_key}' or its parent keys not found in {record_file_path}")
except (SyntaxError, ValueError) as e:
    print(f"Error decoding RTL content literal: {e}")
except Exception as e:
    print(f"An unexpected error occurred loading record data: {e}")

# Load rules text (remains the same)
rules_file_path = project_root / "dev_docs" / "rules.md"
rules_text = load_rules(str(rules_file_path))

if not rules_text:
    print(f"Warning: Could not load rules from {rules_file_path}. Verification steps might fail.")
    rules_text = """## Control Flow Error Detection (CFED) Rules\nVar = #initial_value\nCompare(Var, #BB_ID)\nVar += #random_val\nVar -= #update_val"""

if example_code is not None and rules_text:
    print(f"Loaded PROCESSED RTL code ({len(example_code)} chars) and rules text ({len(rules_text)} chars).")
elif example_code is None:
    print("Error: Example code could not be loaded/processed from record.json. Analysis cannot proceed.")

Loading record data from: /workspace/src/analyzer/record.json
Successfully extracted and decoded RAW RTL content for key: makeStrings/RTL_Protected.txt
--- Raw RTL Snippet ---
BB: -2

----------------------------------------------------------------

BB: 0
(insn 1543 6 1542 2 (parallel [
            (asm_input/v ("STMDB r6!, {r11}") <built-in>:0)
            (clobber (mem:BLK (scratch) [0  A8]))
        ]) -1
     (nil))
(insn 1542 1543 1435 2 (set (reg:SI 11 fp)
        (const_int 323 [0x143])) -1
     (nil))
(insn 1435 1542 1436 2 (set (reg:SI 11 fp)
        (plus:SI (reg:SI 11 fp)
            (const_int -263 [0xfffffffffffffef9]))) -1
     (nil))
(insn 1436 1435 143...
-----------------------
Extracting core instructions from loaded RTL...
Original raw size: 62694 chars, Processed size: 15913 chars
--- Processed RTL Snippet ---
BB: -2
BB: 0
(insn (parallel [
            (asm_input/v ("STMDB r6!, {r11}") <built-in>:0)
(insn (set (reg:SI 11 fp)
        (const_int 323 [0x143])) -1
(insn

## 4. Create Analysis Runnables

Now, we create instances of our LangChain prompts/runnables.

In [11]:
# Create runnables only if LLM is available
graph_runnable: Optional[DistillationPrompt] = None
protection_runnable: Optional[DistillationPrompt] = None
verification_runnable: Optional[DistillationPrompt] = None
reporting_runnable: Optional[DistillationPrompt] = None

if llm:
    try:
        graph_runnable = create_graph_analysis_runnable(llm)
        protection_runnable = create_protection_determination_runnable(llm)
        verification_runnable = create_rule_verification_runnable(llm)
        reporting_runnable = create_reporting_runnable(llm)
        print("Runnables created.")
    except Exception as e:
        print(f"Error creating runnables: {e}")
else:
    print("LLM not initialized. Skipping runnable creation.")

Runnables created.


## 5. Simple RTL Classification Test

Let's isolate the classification. Does the LLM recognize the *raw* code as Verilog/VHDL with a very simple prompt?

In [14]:
# Define a very simple prompt for classification only
simple_rtl_check_template = "Is code look similar to GCC Register Transfer Language format, \n\nCode:\n```hdl\n{code}\n```"
simple_rtl_check_prompt = PromptTemplate.from_template(simple_rtl_check_template)

# Create a simple chain
simple_rtl_check_runnable = None
if llm:
    simple_rtl_check_runnable = simple_rtl_check_prompt | llm | StrOutputParser()
    print("Simple RTL check runnable created.")
else:
    print("LLM not available, skipping simple check runnable creation.")

# Run the simple check on the RAW code
simple_result = "(skipped)"
if simple_rtl_check_runnable and example_code_raw is not None:
    print("Running simple RTL check on RAW code...")
    try:
        simple_result = simple_rtl_check_runnable.invoke({"code": example_code_raw}).strip()
        print(f"---> Simple RTL Check Result: '{simple_result}'")
    except Exception as e:
        print(f"Error during simple RTL check: {e}")
        simple_result = "(error)"
elif not simple_rtl_check_runnable:
    print("Simple check runnable not available.")
elif example_code_raw is None:
    print("Raw example code not available.")

Simple RTL check runnable created.
Running simple RTL check on RAW code...
---> Simple RTL Check Result: 'Yes, the code you've provided appears to be in GCC's Register Transfer Language (RTL) format. RTL is an intermediate representation used by GCC during compilation, representing the program's instructions in a form that's closer to machine code but still architecture-independent.

Key characteristics that identify this as RTL:

1. **Basic Block Structure**: The code is organized into basic blocks (BB: 0, BB: 1, etc.), which is typical of RTL output.

2. **Instruction Format**: Each instruction follows the pattern:
   ```
   (insn [id] [prev] [next] [block#] [RTL expression] [source location] [attributes])
   ```

3. **RTL Expressions**: The instructions contain RTL expressions like:
   - `(set (reg:SI 11 fp) (const_int 323 [0x143]))`
   - `(plus:SI (reg:SI 11 fp) (const_int -263 [0xfffffffffffffef9]))`
   - `(compare:CC (reg:SI 11 fp) (const_int 60 [0x3c]))`

4. **Register Notation*

In [ ]:
# Define a very simple prompt for classification only
simple_rtl_check_template = "Is the following code Verilog or VHDL? Respond ONLY with 'Yes' or 'No'.\n\nCode:\n```hdl\n{code}\n```"
simple_rtl_check_prompt = PromptTemplate.from_template(simple_rtl_check_template)

# Create a simple chain
simple_rtl_check_runnable = None
if llm:
    simple_rtl_check_runnable = simple_rtl_check_prompt | llm | StrOutputParser()
    print("Simple RTL check runnable created.")
else:
    print("LLM not available, skipping simple check runnable creation.")

# Run the simple check on the RAW code
simple_result = "(skipped)"
if simple_rtl_check_runnable and example_code_raw is not None:
    print("Running simple RTL check on RAW code...")
    try:
        simple_result = simple_rtl_check_runnable.invoke({"code": example_code_raw}).strip()
        print(f"---> Simple RTL Check Result: '{simple_result}'")
    except Exception as e:
        print(f"Error during simple RTL check: {e}")
        simple_result = "(error)"
elif not simple_rtl_check_runnable:
    print("Simple check runnable not available.")
elif example_code_raw is None:
    print("Raw example code not available.")

Simple RTL check runnable created.
Running simple RTL check on RAW code...
---> Simple RTL Check Result: 'No'


In [ ]:
# Define a very simple prompt for classification only
simple_rtl_check_template = "Is the following code Verilog or VHDL? Respond ONLY with 'Yes' or 'No'.\n\nCode:\n```hdl\n{code}\n```"
simple_rtl_check_prompt = PromptTemplate.from_template(simple_rtl_check_template)

# Create a simple chain
simple_rtl_check_runnable = None
if llm:
    simple_rtl_check_runnable = simple_rtl_check_prompt | llm | StrOutputParser()
    print("Simple RTL check runnable created.")
else:
    print("LLM not available, skipping simple check runnable creation.")

# Run the simple check on the RAW code
simple_result = "(skipped)"
if simple_rtl_check_runnable and example_code_raw is not None:
    print("Running simple RTL check on RAW code...")
    try:
        simple_result = simple_rtl_check_runnable.invoke({"code": example_code_raw}).strip()
        print(f"---> Simple RTL Check Result: '{simple_result}'")
    except Exception as e:
        print(f"Error during simple RTL check: {e}")
        simple_result = "(error)"
elif not simple_rtl_check_runnable:
    print("Simple check runnable not available.")
elif example_code_raw is None:
    print("Raw example code not available.")

Simple RTL check runnable created.
Running simple RTL check on RAW code...
---> Simple RTL Check Result: 'No'


In [ ]:
# Define a very simple prompt for classification only
simple_rtl_check_template = "Is the following code Verilog or VHDL? Respond ONLY with 'Yes' or 'No'.\n\nCode:\n```hdl\n{code}\n```"
simple_rtl_check_prompt = PromptTemplate.from_template(simple_rtl_check_template)

# Create a simple chain
simple_rtl_check_runnable = None
if llm:
    simple_rtl_check_runnable = simple_rtl_check_prompt | llm | StrOutputParser()
    print("Simple RTL check runnable created.")
else:
    print("LLM not available, skipping simple check runnable creation.")

# Run the simple check on the RAW code
simple_result = "(skipped)"
if simple_rtl_check_runnable and example_code_raw is not None:
    print("Running simple RTL check on RAW code...")
    try:
        simple_result = simple_rtl_check_runnable.invoke({"code": example_code_raw}).strip()
        print(f"---> Simple RTL Check Result: '{simple_result}'")
    except Exception as e:
        print(f"Error during simple RTL check: {e}")
        simple_result = "(error)"
elif not simple_rtl_check_runnable:
    print("Simple check runnable not available.")
elif example_code_raw is None:
    print("Raw example code not available.")

Simple RTL check runnable created.
Running simple RTL check on RAW code...
---> Simple RTL Check Result: 'No'


## 6. Test Graph Analysis Step (Using Processed and Raw Code)

Now, run the full graph analysis prompt (with the `RTL Status:` format) on both the processed code and the raw code to compare.

In [ ]:
graph_result_processed = None # For processed code
graph_result_raw = None      # For raw code

if graph_runnable and example_code is not None and example_code_raw is not None:
    print("--- Running Graph Analysis on PROCESSED Code ---")
    try:
        # Invoke on processed code
        graph_result_processed: GraphAnalysisOutput = graph_runnable.runnable.invoke({"code": example_code})
        print("--- Processed Code Analysis Result Object ---")
        print(graph_result_processed)
        print(f"is_rtl: {graph_result_processed.is_rtl}")
        print(f"analysis_skipped_reason: {graph_result_processed.analysis_skipped_reason}")
        print("-" * 40)

    except Exception as e:
        print(f"Error during Processed Code Graph Analysis invocation: {e}")
        import traceback
        traceback.print_exc()

    print("\n--- Running Graph Analysis on RAW Code ---")
    try:
        # Invoke on raw code
        graph_result_raw: GraphAnalysisOutput = graph_runnable.runnable.invoke({"code": example_code_raw})
        print("--- Raw Code Analysis Result Object ---")
        print(graph_result_raw)
        print(f"is_rtl: {graph_result_raw.is_rtl}")
        print(f"analysis_skipped_reason: {graph_result_raw.analysis_skipped_reason}")
        print("-" * 40)

    except Exception as e:
        print(f"Error during Raw Code Graph Analysis invocation: {e}")
        import traceback
        traceback.print_exc()

elif not graph_runnable:
    print("Graph runnable not created. Skipping analysis.")
elif example_code is None or example_code_raw is None:
    print("Example code (raw or processed) not loaded. Skipping analysis.")

## 7. Interpretation

1.  **Simple Check (Cell 5):** Did the super simple check output `Yes` or `No`? 
    *   If it output `No` (or anything other than `Yes`), the LLM is fundamentally failing to classify the raw RTL code. This could be an LLM limitation or an issue with the specific RTL code snippet. 
    *   If it output `Yes`, then the LLM *can* classify it correctly when the task is simple.
2.  **Graph Analysis Check (Cell 6):**
    *   Compare the results for processed vs. raw code. Is `is_rtl` still `False` for both?
    *   If the simple check was `Yes` but the graph analysis check (for raw code) is `False`, it strongly suggests the complexity of the graph analysis prompt (even the `RTL Status:` version) is confusing the LLM and causing it to fail the classification part.
    *   If *both* the simple check and the graph analysis (for raw code) are `False`, it reinforces the idea that the LLM struggles with this specific code/task.

Based on these results, we can decide whether to further simplify the graph prompt, adjust the `extract_core_instructions` logic (if processed code fails differently), or consider if the current LLM is suitable for this task.

## 8. (Optional) Continue Full Pipeline

If the graph analysis seems correct, you can uncomment and run the following cells to execute the rest of the pipeline (Protection Determination, Rule Verification, Reporting) using the result from the graph analysis step above.

In [ ]:
# protection_result = None
# protection_json = None
# if protection_runnable and graph_result_processed and graph_result_processed.is_rtl and not graph_result_processed.analysis_skipped_reason:
#     print("\nRunning Protection Determination...")
#     try:
#         # Extract the JSON string from the successful graph analysis
#         graph_json_str = graph_result_processed.model_dump_json() if hasattr(graph_result_processed, 'model_dump_json') else graph_result_processed.json()
# 
#         protection_result: ProtectionDeterminationOutput = protection_runnable.runnable.invoke({"graph_json": graph_json_str})
#         protection_json = protection_result.model_dump_json(indent=2) if hasattr(protection_result, 'model_dump_json') else protection_result.json(indent=2)
#         print("Protection Determination Output:")
#         print(protection_json)
#     except Exception as e:
#         print(f"Error during Protection Determination: {e}")
#         protection_result = None 
#         protection_json = None
# elif not protection_runnable:
#      print("Skipping Protection Determination: Runnable not available.")
# else:
#     print("Skipping Protection Determination: Previous graph analysis did not succeed or was not RTL.")

In [ ]:
# verification_result = None
# verification_json = None
# if verification_runnable and graph_result_processed and graph_result_processed.is_rtl and protection_result and example_code and rules_text:
#     print("\nRunning Rule Verification...")
#     try:
#         graph_json_str = graph_result_processed.model_dump_json() if hasattr(graph_result_processed, 'model_dump_json') else graph_result_processed.json()
#         verification_input = {
#             "protection_type": protection_result.protection_type,
#             "code": example_code, # Use processed code
#             "graph_json": graph_json_str,
#             "rules_text": rules_text
#         }
#         verification_result: RuleVerificationOutput = verification_runnable.runnable.invoke(verification_input)
#         verification_json = verification_result.model_dump_json(indent=2) if hasattr(verification_result, 'model_dump_json') else verification_result.json(indent=2)
#         print("Rule Verification Output:")
#         print(verification_json)
#     except Exception as e:
#         print(f"Error during Rule Verification: {e}")
#         verification_json = None
# elif not verification_runnable:
#     print("Skipping Rule Verification: Runnable not available.")
# else:
#     print("Skipping Rule Verification: Previous steps did not succeed or inputs missing.")

In [ ]:
# final_report = None
# report_json = None
# if reporting_runnable and graph_result_processed and graph_result_processed.is_rtl and protection_json and verification_json:
#     print("\nGenerating Final Report...")
#     try:
#         graph_json_str = graph_result_processed.model_dump_json() if hasattr(graph_result_processed, 'model_dump_json') else graph_result_processed.json()
#         reporting_input = {
#             "graph_json": graph_json_str,
#             "protection_determination_json": protection_json,
#             "rule_verification_json": verification_json
#         }
#         final_report: ComplianceReport = reporting_runnable.runnable.invoke(reporting_input)
#         report_json = final_report.model_dump_json(indent=2) if hasattr(final_report, 'model_dump_json') else final_report.json(indent=2)
#         print("Final Compliance Report Output:")
#         print(report_json)
#     except Exception as e:
#         print(f"Error during Reporting: {e}")
# elif not reporting_runnable:
#      print("Skipping Reporting: Runnable not available.")
# else:
#     print("Skipping Reporting: Previous steps did not succeed.")